In [1]:
import subprocess
subprocess.run(["pip", "install", "datasets", "requests"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'datasets', 'requests'], returncode=0, stdout=b'Collecting datasets\n  Downloading datasets-4.8.4-py3-none-any.whl.metadata (19 kB)\nRequirement already satisfied: requests in /Users/benjaminburda/Desktop/Thesis/nyt-connections-thesis/venv/lib/python3.10/site-packages (2.33.0)\nRequirement already satisfied: filelock in /Users/benjaminburda/Desktop/Thesis/nyt-connections-thesis/venv/lib/python3.10/site-packages (from datasets) (3.25.2)\nRequirement already satisfied: numpy>=1.17 in /Users/benjaminburda/Desktop/Thesis/nyt-connections-thesis/venv/lib/python3.10/site-packages (from datasets) (2.2.6)\nCollecting pyarrow>=21.0.0 (from datasets)\n  Downloading pyarrow-23.0.1-cp310-cp310-macosx_12_0_arm64.whl.metadata (3.1 kB)\nCollecting dill<0.4.2,>=0.3.0 (from datasets)\n  Downloading dill-0.4.1-py3-none-any.whl.metadata (10 kB)\nRequirement already satisfied: pandas in /Users/benjaminburda/Desktop/Thesis/nyt-connections-thesis/venv/lib/python3.10/s

In [ ]:
import json
import requests
import pandas as pd

# Eyefyre archive
eyefyre_url = "https://raw.githubusercontent.com/Eyefyre/NYT-Connections-Answers/main/connections.json"
eyefyre_raw = requests.get(eyefyre_url).json()
print(f"Eyefyre: {len(eyefyre_raw)} puzzles loaded")
print("Sample entry:")
print(json.dumps(eyefyre_raw[0], indent=2))

Eyefyre: 1016 puzzles loaded
Sample entry:
{
  "id": 1,
  "date": "2023-06-12",
  "answers": [
    {
      "level": 0,
      "group": "WET WEATHER",
      "members": [
        "HAIL",
        "RAIN",
        "SLEET",
        "SNOW"
      ]
    },
    {
      "level": 1,
      "group": "NBA TEAMS",
      "members": [
        "BUCKS",
        "HEAT",
        "JAZZ",
        "NETS"
      ]
    },
    {
      "level": 2,
      "group": "KEYBOARD KEYS",
      "members": [
        "OPTION",
        "RETURN",
        "SHIFT",
        "TAB"
      ]
    },
    {
      "level": 3,
      "group": "PALINDROMES",
      "members": [
        "KAYAK",
        "LEVEL",
        "MOM",
        "RACECAR"
      ]
    }
  ]
}


In [8]:
lopez_url = "https://huggingface.co/datasets/tm21cy/NYT-Connections/resolve/main/ConnectionsFinalDataset.json"
lopez_raw = requests.get(lopez_url).json()
lopez_df = pd.DataFrame(lopez_raw)
print(f"Lopez: {len(lopez_df)} puzzles loaded")
print("Columns:", lopez_df.columns.tolist())
print("\nSample entry:")
print(json.dumps(lopez_raw[0], indent=2))

Lopez: 652 puzzles loaded
Columns: ['date', 'contest', 'words', 'answers', 'difficulty']

Sample entry:
{
  "date": "2024/06/03",
  "contest": "NYT Connections 358 - June 3rd, 2024",
  "words": [
    "LASER",
    "PLUCK",
    "THREAD",
    "WAX",
    "COIL",
    "SPOOL",
    "WIND",
    "WRAP",
    "HONEYCOMB",
    "ORGANISM",
    "SOLAR PANEL",
    "SPREADSHEET",
    "BALL",
    "MOVIE",
    "SCHOOL",
    "VITAMIN"
  ],
  "answers": [
    {
      "answerDescription": "REMOVE, AS BODY HAIR",
      "words": [
        "LASER",
        "PLUCK",
        "THREAD",
        "WAX"
      ]
    },
    {
      "answerDescription": "TWIST AROUND",
      "words": [
        "COIL",
        "SPOOL",
        "WIND",
        "WRAP"
      ]
    },
    {
      "answerDescription": "THINGS MADE OF CELLS",
      "words": [
        "HONEYCOMB",
        "ORGANISM",
        "SOLAR PANEL",
        "SPREADSHEET"
      ]
    },
    {
      "answerDescription": "B-___",
      "words": [
        "BALL",
        "M

In [ ]:
from datetime import datetime

# Normalize dates
def parse_eyefyre_date(d):
    return datetime.strptime(d, "%Y-%m-%d").date()

def parse_lopez_date(d):
    return datetime.strptime(d, "%Y/%m/%d").date()

# Build Eyefyre dataframe
eyefyre_rows = []
for puzzle in eyefyre_raw:
    row = {
        "date": parse_eyefyre_date(puzzle["date"]),
        "puzzle_id": puzzle["id"],
        "groups": puzzle["answers"]  # list of {level, group, members}
    }
    eyefyre_rows.append(row)
eyefyre_df = pd.DataFrame(eyefyre_rows)

# Build Lopez dataframe
lopez_rows = []
for puzzle in lopez_raw:
    row = {
        "date": parse_lopez_date(puzzle["date"]),
        "difficulty": puzzle["difficulty"],
        "words": puzzle["words"]
    }
    lopez_rows.append(row)
lopez_df_clean = pd.DataFrame(lopez_rows)

# Merge on date
merged_df = eyefyre_df.merge(lopez_df_clean, on="date", how="left")
labeled = merged_df[merged_df["difficulty"].notna()]

print(f"Total puzzles: {len(merged_df)}")
print(f"Puzzles with difficulty scores: {len(labeled)}")
print(f"Date range of labeled puzzles: {labeled['date'].min()} to {labeled['date'].max()}")

Total puzzles: 1016
Puzzles with difficulty scores: 227
Date range of labeled puzzles: 2023-10-16 to 2024-06-03


In [11]:
print(f"Total puzzles: {len(merged_df)}")
print(f"Difficulty null: {merged_df['difficulty'].isna().sum()}")
print(f"Difficulty not null: {merged_df['difficulty'].notna().sum()}")

# also check if any are null within the lopez dataset itself
print(f"\nNull difficulty within Lopez dataset: {lopez_df_clean['difficulty'].isna().sum()}")
print(f"Non-null difficulty within Lopez dataset: {lopez_df_clean['difficulty'].notna().sum()}")

Total puzzles: 1016
Difficulty null: 789
Difficulty not null: 227

Null difficulty within Lopez dataset: 425
Non-null difficulty within Lopez dataset: 227


In [ ]:
import os

# convert date to string for saving
merged_df["date"] = merged_df["date"].astype(str)

# save full merged dataset
os.makedirs("../data", exist_ok=True)
merged_df.to_json("../data/puzzles_merged.json", orient="records", indent=2)

# save labeled subset only
labeled = merged_df[merged_df["difficulty"].notna()].copy()
labeled.to_json("../data/puzzles_labeled.json", orient="records", indent=2)

print(f"Saved {len(merged_df)} puzzles to data/puzzles_merged.json")
print(f"Saved {len(labeled)} labeled puzzles to data/puzzles_labeled.json")

Saved 1016 puzzles to data/puzzles_merged.json
Saved 227 labeled puzzles to data/puzzles_labeled.json
